In [2]:
# Setup: code comes from GitHub onto the VM disk; data + checkpoints live on Drive (see src/config.py)
import os, sys
from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = "/content/s1156052/NeoAntigen"
try:   # private repo: store a GitHub token as Colab secret GH_TOKEN (never write it in the notebook)
    from google.colab import userdata
    TOKEN = userdata.get("GH_TOKEN")
except Exception:
    TOKEN = ""
URL = f"https://{TOKEN + '@' if TOKEN else ''}github.com/Tienanh209/NeoAntigen.git"
if not os.path.exists(REPO_DIR):
    !git clone {URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)
!apt-get -qq install -y cd-hit > /dev/null

Mounted at /content/drive
Cloning into '/content/s1156052/NeoAntigen'...
remote: Enumerating objects: 82, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 82 (delta 37), reused 67 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (82/82), 1.55 MiB | 26.83 MiB/s, done.
Resolving deltas: 100% (37/37), done.
/content/s1156052/NeoAntigen


In [ ]:
# Download once (dated snapshot on Drive): lung-cancer neoantigens (positives) + viral/TAA epitopes (negatives)
import pandas as pd
from src import config, lung
pd.set_option("display.width", 200)

raw = lung.get_raw(config.DATA_DIR, refresh=False)
print(raw.groupby("group").size())
# Optional extra positives, downloaded by hand into DATA_DIR/lung_extra/ (TSNAdb is *predicted*, not validated):
# extra = lung.load_extra(config.DATA_DIR / "lung_extra" / "tsnadb_lung.tsv", "tsnadb", seq_col="peptide", hla_col="HLA")
# raw = lung.add_extra(raw, [extra])     # then use LungPreset(..., extra_sources=("tsnadb",))
raw = lung.add_extra(raw, [])
lung.sweep(raw)

In [ ]:
# Run the full pipeline for one preset; output goes to DATA_DIR/<name>/ (data.load_dataset(config.DATA_DIR / PRESET))
PRESET = "lung_cedar"
df, final, tables = lung.run(raw, lung.PRESETS[PRESET], config.DATA_DIR, config.WORK_DIR)
for t in tables:
    print(t.to_string(index=False), "\n")

In [ ]:
# Guide steps 3-5: frozen splits (conventional independent test + temporal test), similarity checks, Tables 1-2
# Look at positives per first-evidence year BEFORE fixing T (the guide: choose cutoffs before looking at results)
from src import lung_split
df, info = lung.build(raw, lung.PRESETS[PRESET])
print(lung_split.positives_by_year(df).to_string())

T, E = 2022, 2026          # train/CV: first evidence <= T ; temporal test: (T, E]
res = lung_split.make_splits(df, config.DATA_DIR, config.WORK_DIR, T=T, E=E, params={**info, "preset": PRESET})
for k in ("table1", "table2", "similarity"):
    print(res[k].to_string(), "\n")
print("limitations:", res["limitations"])